# FedUP reproduction: start here (Kaggle / Colab)
**Every section is atomic.** After a kernel restart run only *0. Setup*, then any single cell: the settings live on disk (`artifacts/notebook_config.json`), and every run that already finished is skipped, so any cell can be re-run (or interrupted) safely.

Read `docs/FIXES_AND_DEVIATIONS.md` first: it lists where the paper and the released code disagree and what has (not) been verified.

Kaggle: *Settings > Accelerator > GPU T4 x2*, Internet on. Colab: *Runtime > Change runtime type > GPU*.

**Reading the logs.** `[fl] round r/R held-out acc ...` is a progress number only (client scenario: the retained clients' own test splits; otherwise 2000 images of the official test set). The reported metrics (R-A, F-A, 0A, PS, ACC, ASR, MIA) are computed after unlearning.

## 0. Setup (re-run after every restart)
Finds the repo, sets persistent data / artifact folders, installs the extras, prints the GPUs and the saved settings.

In [ ]:
# find the repo root (works from notebooks/<group>/, from the repo root, on Kaggle and on Colab) and set up the folders
import os, sys, shutil
from pathlib import Path
KAGGLE_REPO = Path("/kaggle/input/datasets/sakhiurrahman/fedupds/FedUP_Reproduction")   # your Kaggle dataset (edit if you move it)
def _find_repo():
    for p in [Path.cwd(), *Path.cwd().parents, Path("/kaggle/working/FedUP_Reproduction"), Path("/content/FedUP_Reproduction"), KAGGLE_REPO]:
        if (p / "fedup_repro").exists():
            return p
    kin = Path("/kaggle/input")
    if kin.exists():                                    # any other layout of the input dataset, up to 6 levels deep
        for dp, dn, _ in os.walk(kin):
            if "fedup_repro" in dn:
                return Path(dp)
            if len(Path(dp).relative_to(kin).parts) > 6:
                dn[:] = []
    return None
ROOT = _find_repo()
assert ROOT is not None, "FedUP_Reproduction folder not found: upload it (Kaggle: as a Dataset) or cd into it"
if str(ROOT).startswith("/kaggle/input"):              # input datasets are read-only: work on a copy of the code ...
    SRC, ROOT = ROOT, Path("/kaggle/working") / ROOT.name
    if not ROOT.exists():
        shutil.copytree(SRC, ROOT, ignore=lambda d, names: [n for n in names if n == "raw" and Path(d).name == "data"])
    os.environ["FEDUP_BUNDLED_RAW"] = str(SRC / "data" / "raw")      # ... and LINK (not copy) the datasets that ship with it
os.chdir(ROOT); sys.path.insert(0, str(ROOT))
from fedup_repro import colab
PERSIST_ROOT = None        # Colab + Drive: "/content/drive/MyDrive/fedup_store" keeps data/ and artifacts/ across sessions (mount Drive first)
print("platform:", colab.setup_dirs(PERSIST_ROOT), "| repo:", ROOT)

import subprocess, sys
colab.pip_install("requirements-colab.txt")
import torch
print("torch", torch.__version__, "| cuda:", torch.cuda.is_available(), "| GPUs:", torch.cuda.device_count())
print(colab.gpu_summary())
colab.dataset_status()       # which raw datasets are already in the repo (those are never downloaded again)
colab.show_config()          # the settings saved by section 0b (defaults if you never ran it)

## 0b. Config: every hyper-parameter in one place
The block below lists **all** hyper-parameters of `fedup_repro/config.py` with their defaults and comments (federated setting, FedAvg training, the attack scenarios, the FedUP filter, evaluation, the EraseClient baseline). Edit values, run the cell, and everything that follows uses them. Notes:
* `rounds_per_epoch` is `1` here (50 rounds); the released code runs `2` (100 rounds). Changing anything that affects training (`rounds_per_epoch`, `local_epochs`, `alpha`, `num_clients`, `lr`, ...) retrains the FedAvg models, because the cache key covers it; FedUP-only settings (`out_dim`, `loss_alpha`, `filter_epochs`, `dp_sigma`, `rho`, ...) reuse the cached FedAvg models.
* the ablation experiments overwrite their own swept value; the sweeps themselves are defined in `fedup_repro/experiments.py`.

In [ ]:
# ====== CONFIG: edit anything below, then run this cell. It is saved to artifacts/notebook_config.json and every experiment cell
# ====== reads it from there, so after a kernel restart you only re-run Setup. Only values that differ from the library default are applied.
SEEDS = (0, 1)        # two seeds keep both T4s busy (one worker process per seed); a single seed leaves one GPU idle
PRESET = None         # None = 'repo' (the released code) | "paper" (the paper's wording) | "fuguard" (used by the cross experiment)
QUICK = False         # True = tiny settings for a pipeline check (3 rounds, 20 % of the data)
GPUS = None           # None = every visible GPU | 1 = force a single process

HP = dict(
    # ---- federated setting ------------------------------------------------------------------------------------
    num_clients=10,  # ASSUMPTION: not in the main text (supplement not available); the repo default is 1
    alpha=0.5,  # [paper] Dirichlet 0.5   (repo argparse default 1.0)
    min_client_size=100,  # [repo] least_samples
    proxy_frac=0.2,  # [repo] fraction of each class of each client held out as proxy data (for the MIA shadow model)
    train_ratio_client=0.7,  # [repo] client scenario: train+test merged, then 70/30 per client
    train_ratio_other=0.99,  # [repo] class / sample scenarios: 99/1 per client; evaluation uses the official test set
    data_fraction=1.0,  # < 1 subsamples the raw data (quick runs)
    # ---- federated training -----------------------------------------------------------------------------------
    local_epochs=5,  # [repo README]
    local_batch=64,  # [repo]
    global_epochs=50,  # [repo README]
    rounds_per_epoch=1,  # NOTEBOOK DEFAULT 1 = 50 FedAvg rounds (the README's '50 global epochs'). 2 = what the released code really runs (100 rounds)
    aggregation='uniform',  # [repo] plain mean of client models ("paper": weighted by data size)
    optimizer='sgd_momentum',  # [repo] SGD(momentum 0.9, wd 5e-4); AdamW for text
    momentum=0.9,
    weight_decay=0.0005,
    pretrained=True,  # [repo] torchvision ResNets start from ImageNet weights (not mentioned in the paper)
    augment='static',  # [repo] one-time random crop+flip baked into every CIFAR image, train AND test ("none" = paper-neutral)
    amp=True,  # fp16 autocast on GPU
    text_model='prajjwal1/bert-tiny',  # paper says "TinyBERT"; the repo loads a local 'bert-tiny-local'. The paper's filter-memory column (32 KB at dim 32) implies a 128-d feature = BERT-tiny, not TinyBERT-312
    max_len=128,
    # ---- unlearning scenarios ---------------------------------------------------------------------------------
    forget_clients=[0],  # [repo default] client 0 is the Byzantine / forgotten client
    forget_classes=[0],  # ASSUMPTION (repo default: empty list)
    backdoor_frac=0.2,  # [repo] fraction of each client's samples carrying the trigger (sample scenario)
    sample_target=0,  # [paper] triggered samples are mapped to class 0
    trigger_mode='repo',  # sample scenario: "repo" writes 255 into the normalised pixel; "white" = a real white pixel
    text_trigger_id=1996,  # [repo] token id appended at the end of the sequence
    bd_patch=4,  # client_backdoor scenario = FuGuard protocol (4x4 white patch -> class 9, all samples)
    bd_target=9,
    bd_ratio=1.0,
    bd_trigger='white',  # "white": a real white patch | "repo": FuGuard's released constants (slightly off white; GRAY for EuroSAT)
    # ---- FedUP (filter) ---------------------------------------------------------------------------------------
    out_dim=32,  # [paper] bottleneck dimension 32
    filter_epochs=50,  # [repo] lora_trained_epoch
    filter_batch=10,  # [repo] centroid mini-batch
    filter_reshuffle=False,  # [repo] batches are formed once and replayed in the same order every epoch
    filter_out_relu=True,  # [repo] the adapter ends with a ReLU
    loss_alpha=0.5,  # [paper] alpha = 0.5 (weight of the cross-entropy; 1-alpha for reconstruction)
    centroid_method='cluster',  # "cluster" (KMeans) | "none" (raw features, ablation)
    dp_sigma=None,  # None -> preset ("repo": 0 = no noise, "paper": PAPER_DP_SIGMA)
    # ---- evaluation -------------------------------------------------------------------------------------------
    r_a_style='class_mean',  # [repo] accuracies are averaged over (client, class) cells; "pooled" = plain accuracy
    eval_batch=256,
    mia=True,
    mia_shadow_epochs=1,
    mia_attack_epochs=20,
    mia_max=2000,
    # ---- EraseClient baseline (ported from FuGuard's FU baseline) ---------------------------------------------
    ec_epochs=5,
    ec_lr=0.001,
    ec_distance_threshold=2.2,
    ec_recovery_rounds=3,
    eval_every=5,
    # ---- None = keep the per-benchmark / per-scenario value --------------------------------------------------------
    lr=None,                     # per benchmark: 0.005 (MNIST, CIFAR-10, AG News), 0.01 (CIFAR-100), 0.001 (ConvNet)
    rho=None,                    # centroid ratio per scenario: client 0.8, class 0.1, sample 0.5
)

colab.save_config(seeds=SEEDS, preset=PRESET, quick=QUICK, gpus=GPUS, hp=HP)

## 1. Smoke test (about 1 minute, CPU, synthetic data)
Checks the whole chain: split, FedAvg, FedUP filter, Retrain, EraseClient, metrics, MIA, tables.

In [ ]:
subprocess.run([sys.executable, '-m', 'fedup_repro.smoke'], check=True)

## 2. Datasets (one cell each; any order)
Datasets are stored **inside the repo** (`data/raw`) as fast-loading arrays. A dataset that is already there, or that ships with the Kaggle dataset `FedUP_Reproduction/data/raw`, is **never downloaded again**. Otherwise CIFAR and MNIST come from the Hugging Face hub (much faster than torchvision's mirror). To keep them for the next session, run *8d* and put the zip content into your Kaggle dataset.

In [ ]:
colab.run_script('download_datasets.py', '--only', 'mnist')

In [ ]:
colab.run_script('download_datasets.py', '--only', 'cifar10')

In [ ]:
colab.run_script('download_datasets.py', '--only', 'cifar100')

In [ ]:
colab.run_script('download_datasets.py', '--only', 'agnews')

SVHN and EuroSAT are only needed for the FuGuard-vs-FedUP comparison (section 6 / 7b.2):

In [ ]:
colab.run_script('download_datasets.py', '--only', 'svhn')

In [ ]:
colab.run_script('download_datasets.py', '--only', 'eurosat')

## 2b. Backdoor explorer (interactive)
Pick a dataset, a class and a trigger, then press **random image**: the figure shows the sample **before** and **after** the backdoor is added, plus a zoom on the bottom-right corner and a difference map. The after-image comes from the same code the experiments use.

Triggers: FedUP's single-pixel sample trigger (as released: 255 written in normalised space, or a real white pixel) and FuGuard's k x k white patch. AG News shows the text with the trigger token.

In [ ]:
from fedup_repro import viz
viz.backdoor_explorer()

## 3. Pilot: one benchmark, FedUP only, one seed
The cheapest real run (MNIST + LeNet-5): confirm the numbers look sane before spending hours.

In [ ]:
colab.run_exp('main', benchmarks=['MNIST-LeNet5'], methods=['fedup'], seeds=(0,))
from fedup_repro import aggregate as A
for sc, df in A.table_main('main').items():
    print(sc); display(df)

## 4. Main grid, one benchmark at a time
Per benchmark, per seed: **3 FedAvg trainings** (client, class/clean and sample views) for FedUP + EraseClient, and **3 more** for Retrain (the expensive baseline), so each benchmark has two cells: *(a) FedUP + EraseClient* then *(b) Retrain*. Run them in any order, in different sessions if needed; check *4.6 Progress* at any time.

### 4.1 MNIST-LeNet5

In [ ]:
colab.run_exp("main", benchmarks=['MNIST-LeNet5'], methods=['fedup', 'eraseclient'])

In [ ]:
colab.run_exp("main", benchmarks=['MNIST-LeNet5'], methods=['retrain'])

### 4.2 CIFAR10-ResNet18

In [ ]:
colab.run_exp("main", benchmarks=['CIFAR10-ResNet18'], methods=['fedup', 'eraseclient'])

In [ ]:
colab.run_exp("main", benchmarks=['CIFAR10-ResNet18'], methods=['retrain'])

### 4.3 AGNews-TinyBERT

In [ ]:
colab.run_exp("main", benchmarks=['AGNews-TinyBERT'], methods=['fedup', 'eraseclient'])

In [ ]:
colab.run_exp("main", benchmarks=['AGNews-TinyBERT'], methods=['retrain'])

### 4.4 CIFAR10-Transformer

In [ ]:
colab.run_exp("main", benchmarks=['CIFAR10-Transformer'], methods=['fedup', 'eraseclient'])

In [ ]:
colab.run_exp("main", benchmarks=['CIFAR10-Transformer'], methods=['retrain'])

### 4.5 CIFAR100-ResNet34

In [ ]:
colab.run_exp("main", benchmarks=['CIFAR100-ResNet34'], methods=['fedup', 'eraseclient'])

In [ ]:
colab.run_exp("main", benchmarks=['CIFAR100-ResNet34'], methods=['retrain'])

### 4.6 Progress of the main grid

In [ ]:
display(colab.progress('main'))

## 5. Ablations (one cell each)

In [ ]:
colab.run_exp("ablation_dim")

In [ ]:
colab.run_exp("ablation_alpha")

In [ ]:
colab.run_exp("ablation_dp")

In [ ]:
colab.run_exp("ablation_centroid")

## 6. Cross-protocol: FedUP under FuGuard's setup (not in the FedUP paper)
FuGuard's exact protocol on FuGuard's four datasets: 10 clients (Dirichlet 0.1; CIFAR-100: 1.0), **client 3 is backdoored** with a 4x4 patch (FuGuard's released trigger constants) relabelled to class 9, 3-block ConvNet, Adam 1e-3, 5 local epochs, batch 256, 49 FedAvg rounds, all data used. FedUP then unlearns client 3 in one shot.

These cells use the `fuguard` preset and **ignore the section-0b hyper-parameters** (`sets=[]`) so the protocol stays identical to FuGuard's; pass `sets=[...]` to change something on purpose. SVHN and EuroSAT need their datasets first (section 2). FedUP alone is enough for the comparison in 7b; the last cell adds Retrain and EraseClient.

In [ ]:
colab.run_exp('cross', benchmarks=['SVHN-ConvNet'], methods=['fedup'], preset='fuguard', sets=[])

In [ ]:
colab.run_exp('cross', benchmarks=['EuroSAT-ConvNet'], methods=['fedup'], preset='fuguard', sets=[])

In [ ]:
colab.run_exp('cross', benchmarks=['CIFAR10-ConvNet'], methods=['fedup'], preset='fuguard', sets=[])

In [ ]:
colab.run_exp('cross', benchmarks=['CIFAR100-ConvNet'], methods=['fedup'], preset='fuguard', sets=[])

In [ ]:
# optional: Retrain and EraseClient under the same protocol (slower)
for b in ['SVHN-ConvNet', 'EuroSAT-ConvNet', 'CIFAR10-ConvNet', 'CIFAR100-ConvNet']:
    colab.run_exp('cross', benchmarks=[b], methods=['retrain', 'eraseclient'], preset='fuguard', sets=[])

## 7. Reports (re-run whenever more runs have finished)

In [ ]:
colab.run_script('make_reports.py')
from fedup_repro import aggregate as A
for sc, df in A.table_main('main').items():
    print(sc); display(df)

In [ ]:
from fedup_repro import aggregate as A
display(A.table_vs_paper('main').head(60))

In [ ]:
from fedup_repro import aggregate as A
display(A.table_cost('main')); display(A.table_non_target_loss('main'))

In [ ]:
from fedup_repro import aggregate as A
display(A.table_cross_protocol('cross'))

## 7b. Our results vs the papers
Two comparisons, each in atomic cells (re-run after more runs have finished):
* **7b.1 FedUP: ours vs the FedUP paper**: the main table (R-A, F-A, 0A, PS, MIA), time and communication, the four ablations, and an automatic scoreboard of the paper's claims.
* **7b.2 FuGuard vs FedUP, on the FuGuard paper's protocol**: FuGuard's numbers from the FuGuard paper (plus our own FuGuard reproduction) next to FedUP run by us under that protocol.

### 7b.1 FedUP: ours vs the FedUP paper

In [ ]:
from fedup_repro import comparison as C, plotting as PL
from IPython.display import Image, display
out = C.fedup_vs_paper()
display(out['summary'].round(3))
if len(out['main']):
    display(Image(str(PL.plot_fedup_parity(out['main']))))

In [ ]:
from fedup_repro import comparison as C
m = C.main_vs_paper()
print('cells compared:', len(m))
display(m.sort_values('abs_delta', ascending=False).head(30))

In [ ]:
from fedup_repro import comparison as C
display(C.cost_vs_paper())

In [ ]:
from fedup_repro import comparison as C
for name in ['dim', 'alpha', 'dp', 'centroid']:
    print(name)
    display(getattr(C, name + '_vs_paper')())

In [ ]:
from fedup_repro import comparison as C
display(C.claims_scoreboard())

### 7b.2 FuGuard vs FedUP (FuGuard paper's protocol)
How to read it:
* **ASR** is shown in **FuGuard's definition** (share of the triggered test set predicted as class 9, which includes the untouched class-9 images and therefore has a floor); the **strict** ASR (non-target inputs only) is listed next to it.
* **FedUP has no recovery rounds**, so FuGuard's recovery-round metric has no FedUP counterpart; FedUP is measured right after its one-shot unlearning.
* **MIA** is not compared: the two papers define it differently and FuGuard reports it only in figures.
* **FedUP costs differ in kind**: every retained client uploads class centroids once, while FuGuard needs the target client plus recovery rounds.
* EuroSAT: the train/test split is a fixed 80/20 here (FuGuard splits with the run seed), and FuGuard's released trigger for EuroSAT is gray, which is reproduced on purpose (`bd_trigger="repo"`).
* "FuGuard (our repro)" is the FuGuard reproduction from Weekly Update 2, listed as a sanity reference.

In [ ]:
from fedup_repro import comparison as C
display(C.fuguard_vs_fedup_compact())

In [ ]:
from fedup_repro import comparison as C, plotting as PL
from IPython.display import Image, display
fv = C.fuguard_vs_fedup()
display(fv)
if 'fedup_ACC' in fv and fv['fedup_ACC'].notna().any():
    display(Image(str(PL.plot_fuguard_vs_fedup(fv.dropna(subset=['fedup_ACC'])))))

In [ ]:
# FuGuard's recovery rounds (FedUP has none): paper vs our reproduction
import pandas as pd
from fedup_repro import paper_reference as P
display(pd.DataFrame({'FuGuard RR(0.95), RR(1) paper': P.FUGUARD_RR_PAPER, 'FuGuard RR(0.95), RR(1) our repro': P.FUGUARD_RR_REPRO}))

## 8. Checkpoint zip: download everything, restore it later
Zips the **current `data/` and `artifacts/` folders** (plus `results/` and the saved config) into **one file** whose inside layout is always `data/...`, `artifacts/...`, `results/...`, and downloads it. Switch a group off in the call to shrink the file (`raw_data=False`: the raw datasets are re-downloadable). Run **8a on Kaggle or 8b on Colab**, each is self-contained.

To resume in a new session: run Setup, then *8c Restore* with that zip: cached FedAvg models, finished runs and the config come back, and every cell continues where it stopped.

### 8a. Kaggle: make the zip and get a download link
The zip is written to `/kaggle/working`; click the link to download it from the notebook. It is also kept as notebook output after *Save Version*, and files from `/kaggle/working` can be turned into a Kaggle Dataset to restore from in a later session. (Kaggle limits the output folder to about 20 GB.)

In [ ]:
from IPython.display import display
zip_path = colab.make_checkpoint_zip(raw_data=True, processed_data=True, checkpoints=True, filters=True, features=True, results=True)
display(colab.kaggle_link(zip_path))

### 8b. Colab: make the zip and download it directly
The zip is built in `/content` and sent to your browser. A very large file can make the browser download time out: switch groups off, or copy the zip to Drive instead (`shutil.copy(zip_path, '/content/drive/MyDrive/')`).

In [ ]:
from google.colab import files
zip_path = colab.make_checkpoint_zip(raw_data=True, processed_data=True, checkpoints=True, filters=True, features=True, results=True)
files.download(str(zip_path))

### 8c. Restore a checkpoint zip into the current session
Kaggle: add the zip as a Dataset (or upload it to `/kaggle/working`) and give its path. Colab: run the upload line, or point at a file on Drive. Existing files are kept unless `overwrite=True`.

In [ ]:
ZIP = None   # e.g. '/kaggle/input/my-fedup-checkpoint/fedup_checkpoint_20261106_120000.zip'  or  '/content/drive/MyDrive/fedup_checkpoint_....zip'
# Colab upload instead:  from google.colab import files; up = files.upload(); ZIP = '/content/' + next(iter(up))
assert ZIP, 'set ZIP above'
colab.restore_checkpoint_zip(ZIP, overwrite=False)
colab.status(); display(colab.progress('main'))

### 8d. Dataset bundle for Kaggle (raw + processed data only)
Zips just `data/` so you can put the datasets **inside your Kaggle dataset** once: download the zip, unzip it into `FedUP_Reproduction/` (it contains `data/raw/...` and `data/processed/...`), and upload a new version of the dataset. From then on Setup finds the datasets under `/kaggle/input/.../FedUP_Reproduction/data/raw` and nothing is downloaded.

In [ ]:
zip_path = colab.make_checkpoint_zip(name='fedup_data_bundle.zip', raw_data=True, processed_data=True, checkpoints=False, filters=False, features=False, results=False, config=False)
display(colab.kaggle_link(zip_path)) if colab.platform() == 'kaggle' else print(zip_path)